# v4b continual run (no z-loss) (GPU 0), from scratch
Random model (phase 0) → Hindi → Chinese → Arabic → Hindi → ... for 18 phases (6 cycles).
Same seed as the control, so phase 1 (Hindi from scratch) is identical in both; they differ only from phase 2.
Checkpoints: `runs/v4b_continual_s0/ckpt/`. Rerunning resumes after the last saved phase.

In [1]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # run this cell FIRST in a fresh kernel
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common_v4 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
m = Seq2Seq(Config()); print(f"model: {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M params | vocab {V} | tags {LANG_TAG}"); del m

device: cuda | NVIDIA GeForce RTX 3090
model: 66.8M params | vocab 32000 | tags {'hi': 4, 'zh': 5, 'ar': 6}


In [2]:
SMOKE = False       # True first, then False for the real run
C = RunConfig(run_name="v4b_continual_s0", mode="continual", order=("hi", "zh", "ar"), n_phases=18, seed=0)
if SMOKE:   # tiny phases, ~10 min, separate run folder
    C.run_name += "_smoke"; C.n_phases = 3; C.eval_every = 50; C.warmup_steps = 50
    C.min_steps = 100; C.cooldown_steps = 100; C.max_stable_steps = 300
print(C)

RunConfig(run_name='v4b_continual_s0', mode='continual', order=('hi', 'zh', 'ar'), init_ckpt=None, n_phases=18, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=30.0)


In [3]:
D = load_data(train_langs=["hi", "zh", "ar"])

  hi: 993,016 train / 2,000 val
  zh: 755,626 train / 2,000 val
  ar: 998,000 train / 2,000 val
data loaded in 47s


In [4]:
model = train_run(C, D)

 RUN v4b_continual_s0 | mode continual | 18 phases after phase 0 | device cuda
 phase order: init (random) → hi → zh → ar → hi → zh → ar → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
phase 0: fresh random model (66.8M params)

 PHASE 0 DONE | trained: init | starting model, no training
--------------------------------------------------------------------------------------
   lang      val       Δ |   chrF      Δ |  chrF (unrestricted)  val (unrestr.)
   hi      8.965         |    1.2        |                  0.4          10.510
   zh      9.641         |    0.0        |                  0.0          10.526
   ar      9.196         |    2.7        |                  1.1          10.542

▶ PHASE 1/18 | training hi (993,016 pairs)
  [p01 hi] step  